# Notebook 7 — Clinical Report Generator
**RSNA Intracranial Hemorrhage Detection**

This notebook ties together all pipeline components to generate structured
screening reports for individual CT brain images.

### Report schema (fixed fields, locked phrases)
Each report contains:
- **Screening outcome** — fixed vocabulary, no diagnostic claim
- **Calibrated confidence** — numeric probability + triage band
- **Recommended action** — one of three fixed phrases
- **Grad-CAM overlay** — visual evidence with caveats
- **Disclaimer** — regulatory/safety language

### Required inputs
- NB02 output: `manifest.csv` + `cache/` NPY arrays
- NB03 output: `best_model.pth`, `checkpoint.pth`
- NB06 output: `calibration_params.json`

> The report generator is deterministic — same image always produces the same report.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
import os, json, base64, datetime, uuid, glob as _glob_mod
from io import BytesIO
from pathlib import Path
import numpy as np
import pandas as pd
import cv2                                  # still needed for cv2.resize in overlay + cv2.imwrite
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from IPython.display import display, HTML

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Input paths ──────────────────────────────────────────────────────────
CACHE_INPUT_DIR   = '/kaggle/input/notebooks/harshitghosh/nb02eda'
NPY_CACHE_DIR     = f'{CACHE_INPUT_DIR}/cache'
MANIFEST_PATH     = f'{CACHE_INPUT_DIR}/manifest.csv'
MODEL_PATH        = '/kaggle/input/notebooks/harshitghosh/03nbeda/best_model.pth'
CHECKPOINT_PATH   = '/kaggle/input/notebooks/harshitghosh/03nbeda/checkpoint.pth'
CALIB_PARAMS_PATH = '/kaggle/input/ich-calibration/calibration_params.json'

ARCH        = 'efficientnet_b0'
IMG_SIZE    = 256
SEED        = 42
N_REPORTS   = 10      # number of sample reports to generate and display

REPORTS_DIR = Path('/kaggle/working/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Load normalization stats ────────────────────────────────────────────
_norm_path = os.path.join(CACHE_INPUT_DIR, 'normalization_stats.json')
if os.path.exists(_norm_path):
    with open(_norm_path) as f:
        _norm = json.load(f)
    MEAN = _norm['mean']
    STD  = _norm['std']
    print(f'Dataset normalization: mean={MEAN}, std={STD}')
else:
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    print(f'Using ImageNet defaults: mean={MEAN}, std={STD}')

print(f'Device: {DEVICE}')

In [ ]:
# ── 1. Load calibration parameters ───────────────────────────────────────
with open(CALIB_PARAMS_PATH) as f:
    calib = json.load(f)

TEMPERATURE     = calib['temperature']
BASE_THRESHOLD  = calib['base_threshold']
HIGH_THRESHOLD  = calib['high_threshold']
LOW_THRESHOLD   = calib['low_threshold']

print('Calibration parameters:')
print(json.dumps(calib, indent=2))

In [ ]:
# ── 2. Load model ─────────────────────────────────────────────────────────
def build_model(arch):
    if arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    elif arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    return m


model = build_model(ARCH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
print('Model loaded.')

In [ ]:
# ── 3. Grad-CAM ───────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, arch):
        self.model = model
        self.activations = self.gradients = None
        target = model.features[-1] if arch == 'efficientnet_b0' else model.layer4[-1]
        self._fh = target.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self._bh = target.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def remove(self):
        self._fh.remove(); self._bh.remove()

    def generate(self, img_tensor):
        self.model.zero_grad()
        t = img_tensor.clone().requires_grad_(True)
        self.model(t).squeeze().backward()
        w   = self.gradients.squeeze().mean(dim=(1, 2), keepdim=True)
        cam = torch.relu((w * self.activations.squeeze()).sum(dim=0)).cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


grad_cam = GradCAM(model, ARCH)
val_transform = T.Compose([T.ToPILImage(), T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])


def load_npy(image_id):
    """Load a cached NPY array and return uint8 RGB for display / overlay."""
    path = os.path.join(NPY_CACHE_DIR, f'{image_id}.npy')
    try:
        return np.load(path)              # uint8  [0, 255]
    except Exception:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)


def get_tensor(image_id):
    return val_transform(load_npy(image_id)).unsqueeze(0).to(DEVICE)


def make_overlay(orig_rgb, cam, alpha=0.45):
    cam_r = cv2.resize(cam, (orig_rgb.shape[1], orig_rgb.shape[0]))
    heat  = (cm.jet(cam_r)[:, :, :3] * 255).astype(np.uint8)
    return (alpha * heat + (1 - alpha) * orig_rgb).astype(np.uint8)


print('Grad-CAM & helpers ready.')

In [ ]:
# ── 4. Inference + calibration pipeline ──────────────────────────────────
def infer_single(image_id: str) -> dict:
    """
    Run forward pass, apply temperature calibration, return prediction dict.
    """
    model.train()   # train mode so Grad-CAM gradients flow
    t = get_tensor(image_id)
    with torch.no_grad():
        logit = model(t).squeeze().cpu().item()

    raw_prob  = float(torch.sigmoid(torch.tensor(logit)).item())
    cal_prob  = float(torch.sigmoid(torch.tensor(logit / TEMPERATURE)).item())

    model.train()
    cam = grad_cam.generate(t)
    model.eval()

    return {'logit': logit, 'raw_prob': raw_prob, 'cal_prob': cal_prob, 'cam': cam}


print('Inference pipeline defined.')

In [ ]:
# ── 5. Report schema (fixed vocabulary) ──────────────────────────────────
"""
FIXED VOCABULARY — do not change these strings.
The report module is intentionally restricted to prevent diagnostic claims.
"""

# Outcome phrases (based on threshold)
OUTCOME_POSITIVE = 'Hemorrhage indicator detected'
OUTCOME_NEGATIVE = 'No hemorrhage indicator detected'

# Band labels
BAND_LABELS = {
    'HIGH'  : 'High confidence',
    'MEDIUM': 'Moderate confidence',
    'LOW'   : 'Low confidence',
}

# Triage action phrases (one per band × outcome combination)
TRIAGE_ACTIONS = {
    ('POSITIVE', 'HIGH')  : 'Urgent radiologist review recommended',
    ('POSITIVE', 'MEDIUM'): 'Prioritised radiologist review recommended',
    ('POSITIVE', 'LOW')   : 'Radiologist review recommended',
    ('NEGATIVE', 'HIGH')  : 'Standard workflow — no immediate escalation indicated',
    ('NEGATIVE', 'MEDIUM'): 'Standard workflow — manual review if clinically indicated',
    ('NEGATIVE', 'LOW')   : 'Standard workflow',
}

DISCLAIMER = (
    'This report is produced by an AI-assisted screening tool and does NOT constitute a '
    'medical diagnosis. All screening findings must be reviewed and confirmed by a '
    'qualified, licensed medical professional before any clinical decision is made. '
    'The system is intended solely as a decision-support aid in a screening workflow '
    'and is not cleared for standalone diagnostic use.'
)


def build_report(image_id: str, inference: dict,
                 true_label: int = None) -> dict:
    """
    Build a structured screening report from inference results.
    Enforces fixed vocabulary — never outputs free-form clinical text.
    """
    cal_prob = inference['cal_prob']

    # Band assignment
    if cal_prob >= HIGH_THRESHOLD:
        band = 'HIGH'
    elif cal_prob >= LOW_THRESHOLD:
        band = 'MEDIUM'
    else:
        band = 'LOW'

    # Outcome: based on base_threshold (Youden-optimal from training)
    is_positive = cal_prob >= BASE_THRESHOLD
    outcome_str = OUTCOME_POSITIVE if is_positive else OUTCOME_NEGATIVE
    outcome_key = 'POSITIVE' if is_positive else 'NEGATIVE'

    triage_action = TRIAGE_ACTIONS[(outcome_key, band)]

    # Save Grad-CAM overlay
    orig_rgb = load_npy(image_id)
    overlay  = make_overlay(orig_rgb, inference['cam'])
    cam_save_path = str(REPORTS_DIR / f'{image_id}_gradcam.png')
    cv2.imwrite(cam_save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    report = {
        'report_id'        : f'RPT_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}_{image_id[-8:]}',
        'generated_at'     : datetime.datetime.utcnow().isoformat() + 'Z',
        'image_id'         : image_id,
        'ground_truth_label': int(true_label) if true_label is not None else 'N/A',
        'screening_module' : {
            'version'             : '1.0',
            'architecture'        : ARCH,
            'calibration_method'  : calib['calibration_method'] if 'calibration_method' in calib else calib['method'],
        },
        'prediction': {
            'screening_outcome'      : outcome_str,
            'raw_probability'        : round(inference['raw_prob'], 4),
            'calibrated_probability' : round(cal_prob, 4),
            'confidence_band'        : band,
            'confidence_band_label'  : BAND_LABELS[band],
            'decision_threshold'     : round(BASE_THRESHOLD, 4),
        },
        'triage': {
            'action'  : triage_action,
            'urgency' : 'URGENT' if (is_positive and band == 'HIGH') else 'STANDARD',
        },
        'explainability': {
            'method'     : 'Gradient-weighted Class Activation Mapping (Grad-CAM)',
            'heatmap_path': cam_save_path,
            'note'       : (
                'Highlighted regions indicate areas with greatest influence on the '
                'screening decision. These are not confirmed anatomical findings.'
            ),
        },
        'disclaimer': DISCLAIMER,
    }
    return report


print('Report schema defined.')

In [ ]:
# ── 6. Generate reports for sample images ────────────────────────────────
import random
random.seed(SEED)

manifest = pd.read_csv(MANIFEST_PATH)
val_df   = manifest[manifest['split'] == 'val'].reset_index(drop=True)

# Sample a mix: 5 positive + 5 negative
pos_samples = val_df[val_df['any'] == 1].sample(N_REPORTS // 2, random_state=SEED)
neg_samples = val_df[val_df['any'] == 0].sample(N_REPORTS // 2, random_state=SEED)
sample_df   = pd.concat([pos_samples, neg_samples]).sample(frac=1, random_state=SEED)

all_reports = []
for _, row in sample_df.iterrows():
    inf = infer_single(row['image_id'])
    rep = build_report(row['image_id'], inf, true_label=row['any'])
    all_reports.append(rep)

    # Save individual JSON report
    report_path = REPORTS_DIR / f'{row["image_id"]}_report.json'
    with open(report_path, 'w') as f:
        json.dump({k: v for k, v in rep.items() if k != 'cam'}, f, indent=2)

print(f'{len(all_reports)} reports generated and saved to {REPORTS_DIR}')

In [ ]:
# ── 7. HTML report display ────────────────────────────────────────────────
def img_to_base64(path: str) -> str:
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


BAND_COLORS = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#27ae60'}
URGENCY_COLORS = {'URGENT': '#e74c3c', 'STANDARD': '#2980b9'}


def render_report_html(report: dict) -> str:
    pred   = report['prediction']
    triage = report['triage']
    expl   = report['explainability']
    band   = pred['confidence_band']
    band_color = BAND_COLORS.get(band, '#555')
    urg_color  = URGENCY_COLORS.get(triage['urgency'], '#555')

    # Embed Grad-CAM image as base64
    cam_data = img_to_base64(expl['heatmap_path']) if os.path.exists(expl['heatmap_path']) else ''
    cam_img_tag = f'<img src="data:image/png;base64,{cam_data}" style="height:200px;"/>' if cam_data else '<i>Image not available</i>'

    outcome_color = '#e74c3c' if 'detected' in pred['screening_outcome'].lower() and 'No' not in pred['screening_outcome'] else '#27ae60'

    html = f"""
<div style="border:1px solid #ccc; border-radius:8px; padding:16px; margin:16px 0;
             font-family:Arial,sans-serif; max-width:800px; background:#fafafa;">
  <h3 style="margin:0 0 8px;">Screening Report
    <span style="font-size:0.7em; color:#888; float:right;">{report['report_id']}</span>
  </h3>
  <p style="margin:2px 0; color:#555; font-size:0.85em;">
    Image ID: <code>{report['image_id']}</code> &nbsp;|
    Generated: {report['generated_at']} &nbsp;|
    Ground truth: <b>{'Positive' if report['ground_truth_label'] == 1 else 'Negative' if report['ground_truth_label'] == 0 else 'Unknown'}</b>
  </p>
  <hr style="margin:10px 0;"/>

  <table style="width:100%; border-collapse:collapse;">
    <tr>
      <td style="width:50%; vertical-align:top; padding-right:16px;">
        <h4 style="margin:0 0 8px;">Screening Outcome</h4>
        <div style="background:{outcome_color}; color:white; padding:8px 12px;
                    border-radius:4px; font-size:1.05em; font-weight:bold;">
          {pred['screening_outcome']}
        </div>

        <h4 style="margin:12px 0 6px;">Confidence</h4>
        <table style="font-size:0.9em; width:100%;">
          <tr><td>Raw probability:</td><td><b>{pred['raw_probability']:.4f}</b></td></tr>
          <tr><td>Calibrated probability:</td><td><b>{pred['calibrated_probability']:.4f}</b></td></tr>
          <tr><td>Decision threshold:</td><td><b>{pred['decision_threshold']:.4f}</b></td></tr>
          <tr><td>Confidence band:</td>
              <td><span style="background:{band_color}; color:white; padding:2px 8px;
                               border-radius:3px; font-weight:bold;">
                {pred['confidence_band_label']} ({band})
              </span></td></tr>
        </table>

        <h4 style="margin:12px 0 6px;">Triage Recommendation</h4>
        <div style="border-left:4px solid {urg_color}; padding-left:10px;">
          <span style="color:{urg_color}; font-weight:bold;">{triage['urgency']}</span><br/>
          {triage['action']}
        </div>
      </td>

      <td style="width:50%; vertical-align:top;">
        <h4 style="margin:0 0 8px;">Visual Evidence (Grad-CAM)</h4>
        {cam_img_tag}
        <p style="font-size:0.8em; color:#888; margin:4px 0;">{expl['note']}</p>
      </td>
    </tr>
  </table>

  <hr style="margin:12px 0;"/>
  <p style="font-size:0.75em; color:#888; margin:0; border-left:3px solid #e74c3c; padding-left:8px;">
    <b>DISCLAIMER:</b> {report['disclaimer']}
  </p>
</div>
"""
    return html


print('HTML renderer defined. Displaying reports...')

In [ ]:
# ── 8. Render all sample reports ──────────────────────────────────────────
for report in all_reports:
    display(HTML(render_report_html(report)))

In [ ]:
# ── 9. Batch summary table ────────────────────────────────────────────────
summary_rows = []
for rep in all_reports:
    pred = rep['prediction']
    summary_rows.append({
        'image_id'           : rep['image_id'][-8:],
        'true_label'         : rep['ground_truth_label'],
        'screening_outcome'  : pred['screening_outcome'],
        'cal_prob'           : pred['calibrated_probability'],
        'confidence_band'    : pred['confidence_band'],
        'triage_action'      : rep['triage']['action'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('/kaggle/working/report_summary.csv', index=False)

print('Batch summary:')
display(summary_df)

print()
print('Correct triage (positive true → urgent):')
tp_urgent = summary_df[
    (summary_df['true_label'] == 1) & (summary_df['triage_action'].str.contains('Urgent|Prioritised'))
]
print(f'  {len(tp_urgent)}/{(summary_df["true_label"]==1).sum()} positive cases escalated correctly')

In [ ]:
# ── 10. Cleanup + final output summary ───────────────────────────────────
grad_cam.remove()

import glob
report_files = glob.glob(str(REPORTS_DIR / '*.json'))
print(f'\nTotal JSON reports saved: {len(report_files)}')
print(f'Grad-CAM overlays saved : {len(glob.glob(str(REPORTS_DIR / "*_gradcam.png")))}')
print(f'Summary CSV             : /kaggle/working/report_summary.csv')
print()
print('All notebooks complete. Full pipeline:')
print('  01_eda.ipynb           → Dataset exploration + windowing demo')
print('  02_preprocess_cache.ipynb → DICOM → NPY cache (commit output)')
print('  03_train_session.ipynb    → Training with session chaining')
print('  04_ablations.ipynb        → Architecture + preprocessing + augmentation ablations')
print('  05_gradcam.ipynb          → Grad-CAM heatmaps + occlusion sanity check')
print('  06_calibration.ipynb      → Temperature scaling + 3-band triage system')
print('  07_report.ipynb           → Structured clinical reports (this notebook)')

In [ ]:
# ── HEALTH CHECK — automated output validation ────────────────────────────

errors = []

# Check reports directory
import glob as _glob
json_reports = _glob.glob(str(REPORTS_DIR / '*.json'))
cam_files    = _glob.glob(str(REPORTS_DIR / '*_gradcam.png'))

if len(json_reports) < N_REPORTS:
    errors.append(f'Only {len(json_reports)} JSON reports — expected {N_REPORTS}')

if len(cam_files) < N_REPORTS:
    errors.append(f'Only {len(cam_files)} Grad-CAM overlays — expected {N_REPORTS}')

# Check summary CSV
if not os.path.exists('/kaggle/working/report_summary.csv'):
    errors.append('report_summary.csv is MISSING')

# Validate report structure (sample one)
if json_reports:
    with open(json_reports[0]) as f:
        sample_rep = json.load(f)
    required_keys = ['report_id', 'image_id', 'prediction', 'triage', 'disclaimer']
    for k in required_keys:
        if k not in sample_rep:
            errors.append(f'Report missing key: {k}')
    if 'disclaimer' in sample_rep and len(sample_rep['disclaimer']) < 50:
        errors.append('Disclaimer text is too short — possible truncation')

health = {
    'notebook'     : '07_report',
    'status'       : 'PASS' if not errors else 'FAIL',
    'errors'       : errors,
    'n_reports'    : len(json_reports),
    'n_gradcam'    : len(cam_files),
    'summary_csv'  : os.path.exists('/kaggle/working/report_summary.csv'),
}

with open('/kaggle/working/health_check_nb07.json', 'w') as f:
    json.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Reports   : {len(json_reports)} JSON + {len(cam_files)} Grad-CAM')
    print(f'   Summary   : saved')
    print(f'   Pipeline  : COMPLETE ✓')
    print()
    print('All 7 notebooks have completed successfully.')
    print('Check health_check_nb*.json files for per-notebook validation results.')